# Model Training

This notebook is optimized for the final deadline sprint.

It does four things:

1. builds a strict grouped evaluation pipeline using the existing `Region` column,
2. tests a small shortlist of target-specific candidates,
3. freezes a safe manifest from full grouped CV,
4. writes three submission files:
   - **A** = safe anchor,
   - **B** = EC aggressive + DRP safe,
   - **C** = hedge blend.

Notes:
- We assume `Region` already exists in the provided dataset.
- We keep the notebook cell-by-cell and avoid one giant integrated script.
- We clip predictions to nonnegative values before submission.

In [52]:
import os
import sys
import json
import time
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from IPython.display import display

## Environment and MLflow

In [53]:
sys.path.append(os.path.abspath('..'))

ENV = 'local'   # switch to 'snowflake' if needed

if ENV == 'local':
    from src import config_local as config
else:
    from src import config_snowflake as config

mlflow.set_tracking_uri(config.MLFLOW_URI)
mlflow.set_experiment('WaterQuality')

print('MLflow URI:', config.MLFLOW_URI)

MLflow URI: sqlite:///../mlflow.db


## Global config

This cell defines:
- targets,
- split metadata,
- artifact directory,
- hashing helpers,
- submission integrity checks.

In [54]:
TARGET_COLS = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus'
]

SPLIT_STRATEGY = 'LORO'
GROUP_DEFINITION_VERSION = 'region_v1'
PIPELINE_VERSION = 'deadline_v2_patch'
PREPROCESS_VERSION = 'median_scaler'
ARTIFACT_DIR = '../models/final_deadline'

os.makedirs(ARTIFACT_DIR, exist_ok=True)


def hash_str(s: str) -> str:
    '''
    Create a short stable hash from a string.
    '''
    return hashlib.sha256(s.encode('utf-8')).hexdigest()[:16]


def hash_list(values) -> str:
    '''
    Hash a list of values after converting to strings.
    '''
    return hash_str('||'.join(map(str, values)))


def compute_group_values_hash(groups: pd.Series) -> str:
    '''
    Hash the exact ordered group assignments.
    Useful to ensure runs are truly comparable.
    '''
    return hash_list(groups.fillna('NA').astype(str).tolist())


def compute_feature_set_hash(features: list) -> str:
    '''
    Hash a feature list in sorted form.
    '''
    return hash_list(sorted(features))


def target_key(target_name: str) -> str:
    '''
    Make a target name filename-safe.
    '''
    return target_name.replace(' ', '')


def make_row_id_template(template_df: pd.DataFrame) -> pd.DataFrame:
    '''
    Add an immutable row_id to the submission template
    so row order can be validated before saving.
    '''
    out = template_df.copy()
    out['row_id'] = np.arange(len(out), dtype=int)
    return out


def assert_submission_integrity(sub_df: pd.DataFrame, template_df: pd.DataFrame, target_cols: list):
    '''
    Validate that the submission is structurally safe.
    '''
    if len(sub_df) != len(template_df):
        raise RuntimeError(f'Row count mismatch: sub={len(sub_df)} template={len(template_df)}')

    if 'row_id' not in sub_df.columns or 'row_id' not in template_df.columns:
        raise RuntimeError('row_id missing in submission/template.')

    if sub_df['row_id'].duplicated().any():
        raise RuntimeError('Duplicate row_id in submission.')

    if not sub_df['row_id'].equals(template_df['row_id']):
        raise RuntimeError('row_id order mismatch.')

    if sub_df[target_cols].isnull().any().any():
        raise RuntimeError('NaN found in target predictions.')

    if (sub_df[target_cols] < 0).any().any():
        raise RuntimeError('Negative predictions found.')


print('Global config loaded.')

Global config loaded.


## Data loading

We load the training data and confirm that the `Region` column already exists.

In [55]:
df = config.load_data()

if 'Region' not in df.columns:
    raise RuntimeError('Region column is required for LORO but was not found.')

if df['Region'].nunique() < 2:
    raise RuntimeError('Need at least 2 unique regions for grouped CV.')

print('Training shape:', df.shape)
print('\nRegion counts:')
print(df['Region'].value_counts(dropna=False))

Training shape: (9319, 32)

Region counts:
Region
Northern_Bulk    5835
Eastern_Cape     1854
Western_Cape     1230
Karoo_Buffer      400
Name: count, dtype: int64


In [56]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9319 entries, 0 to 9318
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Total Alkalinity                9319 non-null   float64
 1   Electrical Conductance          9319 non-null   float64
 2   Dissolved Reactive Phosphorus   9319 non-null   float64
 3   Region                          9319 non-null   object 
 4   nir                             9319 non-null   float64
 5   green                           9319 non-null   float64
 6   swir16                          9319 non-null   float64
 7   swir22                          9319 non-null   float64
 8   NDMI                            9319 non-null   float64
 9   MNDWI                           9319 non-null   float64
 10  pet                             9319 non-null   float64
 11  elevation_meters                9319 non-null   float64
 12  total_precipitation             93

## Feature engineering

This notebook only uses the two engineered features that were explicitly confirmed from the winning setup:

- `pop_density_upstream`
- `specific_discharge`

In [57]:
def safe_divide(num, den, eps=1e-8):
    '''
    Divide safely and return NaN when the denominator is effectively zero.
    '''
    num = pd.Series(num, copy=False).astype(float)
    den = pd.Series(den, copy=False).astype(float)
    out = np.where(np.abs(den) < eps, np.nan, num / den)
    return pd.Series(out, index=num.index)


def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    '''
    Create only the confirmed engineered features from the winning setup.
    '''
    df_eng = data.copy()

    df_eng['pop_density_upstream'] = (
        df_eng['worldpop_mean_1km'] /
        (df_eng['basin_upstream_area_km2'] + 1e-5)
    )

    df_eng['specific_discharge'] = (
        df_eng['river_avg_discharge_cms'] /
        (df_eng['basin_upstream_area_km2'] + 1e-5)
    )

    df_eng[['pop_density_upstream', 'specific_discharge']] = (
        df_eng[['pop_density_upstream', 'specific_discharge']]
        .replace([np.inf, -np.inf], np.nan)
    )

    return df_eng

df = engineer_features(df)

print('Engineered features added')

Engineered features added


## Frozen feature sets

We use two frozen feature sets:

- **A** = stable base set
- **B** = base + empirical interactions

In [58]:
BASE_FEATURES = [
    'nir', 'green', 'swir22', 'NDMI', 'MNDWI', 'pet', 'elevation_meters',
    'total_precipitation', 'average_wind_speed',
    'soil_phh2o_mean_0_5cm', 'soil_clay_mean_0_5cm', 'soil_sand_mean_0_5cm',
    'soil_silt_mean_0_5cm', 'soil_cec_mean_0_5cm',
    'sanlc2022_impact_1km', 'sanlc2020_impact_1km', 'sanlc_change_2020_2022',
    'worldpop_mean_1km', 'basin_upstream_area_km2', 'river_avg_discharge_cms'
]

ENGINEERED_CONFIRMED = [
    'pop_density_upstream',
    'specific_discharge'
]

# Benchmark-like local signal set
BENCHMARK_4 = ['swir22', 'NDMI', 'MNDWI', 'pet']

FEATURE_SETS = {
    'A': BASE_FEATURES,
    'B': BASE_FEATURES + ENGINEERED_CONFIRMED,
    'C': BENCHMARK_4,
    'D': BENCHMARK_4 + ['nir', 'green'],
}

def features_for_set(df_local: pd.DataFrame, fs_name: str):
    '''
    Return a strict feature list for a named feature set.
    Fail fast if expected columns are missing.
    '''
    requested = FEATURE_SETS[fs_name]
    missing = [f for f in requested if f not in df_local.columns]
    if missing:
        raise RuntimeError(f'Missing in {fs_name}: {missing}')
    return requested

print('Feature sets ready:')
for k, v in FEATURE_SETS.items():
    print(f'{k}: {len(v)} features')


Feature sets ready:
A: 20 features
B: 22 features
C: 4 features
D: 6 features


## Preprocessing

We use median imputation and standard scaling inside the CV pipeline.

In [59]:
def get_preprocessor(features_used):
    '''
    Build the preprocessing pipeline for numeric features.
    '''
    numeric_pipe = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    return ColumnTransformer(
        transformers=[('num', numeric_pipe, features_used)],
        remainder='drop'
    )

## Models and shortlist

This shortlist is deliberately small:
- TA: mostly stable XGB
- EC: linear empirical + one XGB challenger
- DRP: safer linear options + one shallow XGB challenger

In [ ]:
from sklearn.ensemble import RandomForestRegressor

def log_wrap(model):
    '''
    Wrap a regressor with log1p / expm1 target transformation.
    '''
    return TransformedTargetRegressor(
        regressor=model,
        func=np.log1p,
        inverse_func=np.expm1
    )

# Centralized defaults (easy to tune in one place)
DEFAULT_XGB_PARAMS = {
    'objective': 'reg:squarederror',
    'n_estimators': 300,
    'learning_rate': 0.03,
    'max_depth': 4,
    'min_child_weight': 10,
    'subsample': 0.70,
    'colsample_bytree': 0.70,
    'reg_alpha': 1.0,
    'reg_lambda': 6.0,
    'random_state': 42,
    'n_jobs': -1,
}

DEFAULT_RF_PARAMS = {
    'n_estimators': 600,
    'min_samples_leaf': 3,
    'max_features': 'sqrt',
    'random_state': 42,
    'n_jobs': -1,
}

MODEL_SPECS = {
    'Ridge_a10_Log': {'kind': 'ridge', 'params': {'alpha': 10.0}, 'log_target': True},
    'Ridge_a30_Log': {'kind': 'ridge', 'params': {'alpha': 30.0}, 'log_target': True},
    'Lasso_a0.0005_Log': {'kind': 'lasso', 'params': {'alpha': 0.0005, 'max_iter': 20000}, 'log_target': True},
    'Lasso_a0.0010_Log': {'kind': 'lasso', 'params': {'alpha': 0.0010, 'max_iter': 20000}, 'log_target': True},
    'Elastic_a0.001_l07_Log': {'kind': 'elastic', 'params': {'alpha': 0.001, 'l1_ratio': 0.7, 'max_iter': 20000, 'random_state': 42}, 'log_target': True},
    'XGB_d4_lr003_Log': {'kind': 'xgb', 'params': {'n_estimators': 300, 'learning_rate': 0.03, 'max_depth': 4}, 'log_target': True},
    'XGB_d3_lr003_Log': {'kind': 'xgb', 'params': {'n_estimators': 220, 'learning_rate': 0.03, 'max_depth': 3}, 'log_target': True},
    'RF_n600_raw': {'kind': 'rf', 'params': {}, 'log_target': False},
    'RF_n600_Log': {'kind': 'rf', 'params': {}, 'log_target': True},
}

def build_model_from_spec(spec):
    kind = spec['kind']
    params = spec.get('params', {})
    use_log = spec.get('log_target', True)

    if kind == 'ridge':
        base = Ridge(**params)
    elif kind == 'lasso':
        base = Lasso(**params)
    elif kind == 'elastic':
        base = ElasticNet(**params)
    elif kind == 'xgb':
        p = DEFAULT_XGB_PARAMS.copy()
        p.update(params)
        base = XGBRegressor(**p)
    elif kind == 'rf':
        p = DEFAULT_RF_PARAMS.copy()
        p.update(params)
        base = RandomForestRegressor(**p)
    else:
        raise ValueError(f'Unknown model kind: {kind}')

    return log_wrap(base) if use_log else base

MODEL_BANK = {name: build_model_from_spec(spec) for name, spec in MODEL_SPECS.items()}

# Feature-set fallback so this cell does not break if C/D are not defined
FS_SMALL = 'C' if 'C' in FEATURE_SETS else 'A'
FS_FULL = 'B' if 'B' in FEATURE_SETS else FS_SMALL
FS_BASE = 'A' if 'A' in FEATURE_SETS else FS_FULL

TARGET_SWEEP = {
    'Total Alkalinity': [
        (FS_SMALL, 'RF_n600_raw'),
        (FS_SMALL, 'RF_n600_Log'),
        (FS_FULL, 'XGB_d4_lr003_Log'),
        (FS_BASE, 'Ridge_a10_Log'),
    ],
    'Electrical Conductance': [
        (FS_SMALL, 'RF_n600_raw'),
        (FS_SMALL, 'RF_n600_Log'),
        (FS_FULL, 'XGB_d4_lr003_Log'),
        (FS_FULL, 'Elastic_a0.001_l07_Log'),
    ],
    'Dissolved Reactive Phosphorus': [
        (FS_SMALL, 'RF_n600_raw'),
        (FS_SMALL, 'RF_n600_Log'),
        (FS_FULL, 'XGB_d3_lr003_Log'),
        (FS_FULL, 'Lasso_a0.0010_Log'),
    ],
}

display(pd.DataFrame(
    [(t, fs, m) for t, recipes in TARGET_SWEEP.items() for fs, m in recipes],
    columns=['target', 'feature_set', 'model']
))

,target,feature_set,model
0,Total Alkalinity,C,RF_n600_raw
1,Total Alkalinity,C,RF_n600_Log
2,Total Alkalinity,B,XGB_d4_lr003_Log
3,Total Alkalinity,A,Ridge_a10_Log
4,Electrical Conductance,C,RF_n600_raw
5,Electrical Conductance,C,RF_n600_Log
6,Electrical Conductance,B,XGB_d4_lr003_Log
7,Electrical Conductance,B,Elastic_a0.001_l07_Log
8,Dissolved Reactive Phosphorus,C,RF_n600_raw
9,Dissolved Reactive Phosphorus,C,RF_n600_Log


## Grouped evaluation helpers

This cell:
- runs grouped out-of-fold predictions,
- reports worst-region behavior,
- saves final artifacts for finalists.

In [71]:
def grouped_oof_eval(df_local, target, estimator, features_used, allowed_regions=None):
    '''
    Run grouped OOF evaluation with LeaveOneGroupOut.

    Returns:
    - full OOF predictions
    - global metrics
    - fold-by-fold region table
    - Eastern_Cape fold R2 if present
    '''
    d = df_local.copy()

    if allowed_regions is not None:
        d = d[d['Region'].isin(allowed_regions)].copy()

    X = d[features_used]
    y = d[target].astype(float).reset_index(drop=True)
    groups = d['Region'].astype(str).reset_index(drop=True)

    logo = LeaveOneGroupOut()

    pipe = Pipeline([
        ('preprocessor', get_preprocessor(features_used)),
        ('model', clone(estimator))
    ])

    pred = cross_val_predict(
        pipe,
        X,
        y,
        groups=groups,
        cv=logo,
        n_jobs=-1,
        verbose=0
    )

    pred = pd.Series(pred).reset_index(drop=True)

    fold_rows = []
    for group_name in groups.unique():
        mask = groups == group_name
        if mask.sum() >= 2:
            fold_rows.append({
                'group': group_name,
                'n': int(mask.sum()),
                'r2': float(r2_score(y[mask], pred[mask])),
                'rmse': float(np.sqrt(mean_squared_error(y[mask], pred[mask]))),
                'mae': float(mean_absolute_error(y[mask], pred[mask])),
            })

    fold_df = pd.DataFrame(fold_rows)

    eastern_cape_r2 = np.nan
    if not fold_df.empty and 'Eastern_Cape' in fold_df['group'].values:
        eastern_cape_r2 = float(
            fold_df.loc[fold_df['group'] == 'Eastern_Cape', 'r2'].iloc[0]
        )

    return {
        'pred': pred.values,
        'rmse': float(np.sqrt(mean_squared_error(y, pred))),
        'mae': float(mean_absolute_error(y, pred)),
        'r2': float(r2_score(y, pred)),
        'mean_fold_r2': float(fold_df['r2'].mean()) if not fold_df.empty else np.nan,
        'min_fold_r2': float(fold_df['r2'].min()) if not fold_df.empty else np.nan,
        'eastern_cape_r2': eastern_cape_r2,
        'fold_df': fold_df,
        'n_rows': int(len(d)),
        'n_groups': int(groups.nunique()),
    }

GLOBAL_R2_WEIGHT = 0.35
EASTERN_CAPE_WEIGHT = 0.50
MIN_FOLD_R2_WEIGHT = 0.15

def compute_selection_score(overall_r2, eastern_cape_r2, min_fold_r2=None):
    '''
    Build a finalist selection score with Eastern_Cape emphasis,
    but keep global performance and fold robustness in the objective.
    '''
    east_term = 0.0 if pd.isna(eastern_cape_r2) else float(eastern_cape_r2)
    min_fold_term = 0.0 if pd.isna(min_fold_r2) else float(min_fold_r2)

    return float(
        GLOBAL_R2_WEIGHT * float(overall_r2) +
        EASTERN_CAPE_WEIGHT * east_term +
        MIN_FOLD_R2_WEIGHT * min_fold_term
    )

def fit_full_and_save(df_local, target, estimator, features_used, run_name):
    '''
    Fit the final full-data pipeline and save preprocessor + model artifacts.
    '''
    X = df_local[features_used]
    y = df_local[target].astype(float)

    pre = get_preprocessor(features_used)
    Xp = pre.fit_transform(X)

    mdl = clone(estimator)
    mdl.fit(Xp, y)

    preproc_path = os.path.join(ARTIFACT_DIR, f'{run_name}__preproc.joblib')
    model_path = os.path.join(ARTIFACT_DIR, f'{run_name}__model.joblib')

    joblib.dump(pre, preproc_path)
    joblib.dump(mdl, model_path)

    return preproc_path, model_path

## Stage 1: Scout run

We first run only a narrow shortlist, optionally prioritizing the hardest regions.

In [72]:
preferred_scout = ['Eastern_Cape', 'Western_Cape', 'Karoo_Buffer']
available_regions = set(df['Region'].astype(str).unique().tolist())
SCOUT_REGIONS = [r for r in preferred_scout if r in available_regions]

if len(SCOUT_REGIONS) < 2:
    SCOUT_REGIONS = None

print('Scout regions:', SCOUT_REGIONS if SCOUT_REGIONS is not None else 'ALL')

rows_scout = []

for target in TARGET_COLS:
    for feature_set_name, model_name in TARGET_SWEEP[target]:
        features = features_for_set(df, feature_set_name)
        estimator = clone(MODEL_BANK[model_name])
        run_name = f'SCOUT__{model_name}__{feature_set_name}__{target_key(target)}'

        print(f'\n--- {run_name} ---')

        with mlflow.start_run(run_name=run_name):
            t0 = time.time()

            out = grouped_oof_eval(
                df_local=df,
                target=target,
                estimator=estimator,
                features_used=features,
                allowed_regions=SCOUT_REGIONS
            )

            dt = time.time() - t0

            mlflow.log_param('stage', 'scout')
            mlflow.log_param('target', target)
            mlflow.log_param('model_name', model_name)
            mlflow.log_param('feature_set_name', feature_set_name)
            mlflow.log_param('split_strategy', SPLIT_STRATEGY)
            mlflow.log_param('group_definition_version', GROUP_DEFINITION_VERSION)
            mlflow.log_param('group_values_hash', compute_group_values_hash(df['Region'].astype(str)))
            mlflow.log_param('feature_set_hash', compute_feature_set_hash(features))
            mlflow.log_param('pipeline_version', PIPELINE_VERSION)
            mlflow.log_param('preprocess_version', PREPROCESS_VERSION)
            mlflow.log_param('n_features_used', len(features))

            mlflow.log_metric('r2', out['r2'])
            mlflow.log_metric('mean_fold_r2', out['mean_fold_r2'])
            mlflow.log_metric('min_fold_r2', out['min_fold_r2'])
            mlflow.log_metric('rmse', out['rmse'])
            mlflow.log_metric('mae', out['mae'])
            mlflow.log_metric('cv_time_sec', dt)

            selection_score = compute_selection_score(
                overall_r2=out['r2'],
                eastern_cape_r2=out['eastern_cape_r2'],
                min_fold_r2=out['min_fold_r2']
            )

            rows_scout.append({
                'stage': 'scout',
                'run_name': run_name,
                'target': target,
                'model_name': model_name,
                'feature_set': feature_set_name,
                'features_used_json': json.dumps(features),
                'r2': out['r2'],
                'mean_fold_r2': out['mean_fold_r2'],
                'min_fold_r2': out['min_fold_r2'],
                'eastern_cape_r2': out['eastern_cape_r2'],
                'selection_score': selection_score,
                'rmse': out['rmse'],
                'mae': out['mae'],
                'cv_time_sec': dt,
            })

        print(out['fold_df'])
        print(
            f"OOF R2={out['r2']:.4f} | "
            f"Eastern_Cape R2={out['eastern_cape_r2']:.4f} | "
            f"selection_score={selection_score:.4f} | "
            f"min_fold_r2={out['min_fold_r2']:.4f}"
        )

scout_df = pd.DataFrame(rows_scout).sort_values(
    ['target', 'selection_score', 'r2', 'min_fold_r2'],
    ascending=[True, False, False, False]
).reset_index(drop=True)

print('\nScout results:')
display(scout_df)

Scout regions: ['Eastern_Cape', 'Western_Cape', 'Karoo_Buffer']

--- SCOUT__RF_n600_raw__C__TotalAlkalinity ---
          group     n        r2       rmse        mae
0  Western_Cape  1230 -1.006131  97.954786  85.752670
1  Eastern_Cape  1854 -0.187456  80.071240  52.269109
2  Karoo_Buffer   400 -0.198333  64.111885  49.580732
OOF R2=-0.3371 | Eastern_Cape R2=-0.1875 | selection_score=-0.3626 | min_fold_r2=-1.0061

--- SCOUT__RF_n600_Log__C__TotalAlkalinity ---
          group     n        r2       rmse        mae
0  Western_Cape  1230 -0.603038  87.562467  75.843932
1  Eastern_Cape  1854 -0.593538  92.757519  61.753577
2  Karoo_Buffer   400 -1.058234  84.022805  63.653265
OOF R2=-0.4887 | Eastern_Cape R2=-0.5935 | selection_score=-0.6266 | min_fold_r2=-1.0582

--- SCOUT__XGB_d4_lr003_Log__B__TotalAlkalinity ---
          group     n        r2       rmse        mae
0  Western_Cape  1230 -0.096505  72.418813  62.765330
1  Eastern_Cape  1854 -0.282634  83.218378  55.860744
2  Karoo_Buffer

,stage,run_name,target,model_name,feature_set,features_used_json,r2,mean_fold_r2,min_fold_r2,eastern_cape_r2,selection_score,rmse,mae,cv_time_sec
0,scout,SCOUT__XGB_d3_lr003_Log__B__DissolvedReactiveP...,Dissolved Reactive Phosphorus,XGB_d3_lr003_Log,B,"[""nir"", ""green"", ""swir22"", ""NDMI"", ""MNDWI"", ""p...",1.928569e-02,3.106312e-03,-1.775076e-02,0.032403,2.028908e-02,29.024043,15.048965,0.340580
1,scout,SCOUT__RF_n600_Log__C__DissolvedReactivePhosph...,Dissolved Reactive Phosphorus,RF_n600_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",4.187678e-03,-5.044937e-02,-1.780296e-01,-0.011988,-3.123288e-02,29.246601,14.244630,1.598629
2,scout,SCOUT__RF_n600_raw__C__DissolvedReactivePhosph...,Dissolved Reactive Phosphorus,RF_n600_raw,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",1.041819e-03,-2.372404e-01,-7.456498e-01,0.040183,-9.139131e-02,29.292760,16.905695,1.599488
3,scout,SCOUT__Lasso_a0.0010_Log__B__DissolvedReactive...,Dissolved Reactive Phosphorus,Lasso_a0.0010_Log,B,"[""nir"", ""green"", ""swir22"", ""NDMI"", ""MNDWI"", ""p...",-3.627344e-01,-4.685743e-01,-9.035177e-01,-0.080597,-3.027831e-01,34.213086,16.807273,0.057012
4,scout,SCOUT__XGB_d4_lr003_Log__B__ElectricalConductance,Electrical Conductance,XGB_d4_lr003_Log,B,"[""nir"", ""green"", ""swir22"", ""NDMI"", ""MNDWI"", ""p...",-8.958836e-02,-2.112636e-01,-2.878293e-01,-0.287829,-2.184450e-01,349.027366,244.983104,0.516032
5,scout,SCOUT__RF_n600_Log__C__ElectricalConductance,Electrical Conductance,RF_n600_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-3.398224e-01,-4.991850e-01,-8.834415e-01,-0.883441,-6.931748e-01,387.036448,288.566064,1.930566
6,scout,SCOUT__RF_n600_raw__C__ElectricalConductance,Electrical Conductance,RF_n600_raw,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-4.781702e-01,-7.116433e-01,-1.760510e+00,-1.760510,-1.311691e+00,406.528012,326.258551,1.586217
7,scout,SCOUT__Elastic_a0.001_l07_Log__B__ElectricalCo...,Electrical Conductance,Elastic_a0.001_l07_Log,B,"[""nir"", ""green"", ""swir22"", ""NDMI"", ""MNDWI"", ""p...",-3.337205e+05,-6.842904e+05,-2.052867e+06,-3.468275,-4.247340e+05,193161.399424,19646.163910,0.046153
8,scout,SCOUT__RF_n600_raw__C__TotalAlkalinity,Total Alkalinity,RF_n600_raw,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-3.370508e-01,-4.639731e-01,-1.006131e+00,-0.187456,-3.626152e-01,85.271394,63.781575,4.392322
9,scout,SCOUT__XGB_d4_lr003_Log__B__TotalAlkalinity,Total Alkalinity,XGB_d4_lr003_Log,B,"[""nir"", ""green"", ""swir22"", ""NDMI"", ""MNDWI"", ""p...",-1.720505e-01,-5.016001e-01,-1.125661e+00,-0.282634,-3.703839e-01,79.836699,60.390951,2.696024


## Stage 2: Full finalists

For each target, we take the top 2 scout candidates and run full grouped CV.
We also save inference artifacts for those finalists.

In [73]:
def build_finalist_shortlist(scout_df_local, top_n=2):
    '''
    Keep a union of top candidates by complementary views so we do not
    discard globally stronger or more robust models too early.
    '''
    pieces = []

    for target_name, tdf in scout_df_local.groupby('target'):
        by_selection = tdf.sort_values(
            ['selection_score', 'r2', 'min_fold_r2'],
            ascending=[False, False, False]
        ).head(top_n)

        by_global = tdf.sort_values(
            ['r2', 'min_fold_r2', 'selection_score'],
            ascending=[False, False, False]
        ).head(top_n)

        by_robust = tdf.sort_values(
            ['min_fold_r2', 'r2', 'selection_score'],
            ascending=[False, False, False]
        ).head(1)

        shortlist = pd.concat([by_selection, by_global, by_robust], axis=0)
        shortlist = shortlist.drop_duplicates(subset=['run_name']).reset_index(drop=True)
        pieces.append(shortlist)

    return pd.concat(pieces, axis=0).reset_index(drop=True)

finalist_df = build_finalist_shortlist(scout_df, top_n=2)

print('Finalists (union shortlist):')
display(finalist_df[[
    'target',
    'feature_set',
    'model_name',
    'selection_score',
    'eastern_cape_r2',
    'r2',
    'min_fold_r2',
    'run_name'
]])

rows_full = []
OOF_PREDS = {}

for _, row in finalist_df.iterrows():
    target = row['target']
    model_name = row['model_name']
    feature_set_name = row['feature_set']
    features = json.loads(row['features_used_json'])
    estimator = clone(MODEL_BANK[model_name])

    run_name = f'FULL__{model_name}__{feature_set_name}__{target_key(target)}'
    print(f'\n=== {run_name} ===')

    with mlflow.start_run(run_name=run_name):
        t0 = time.time()

        out = grouped_oof_eval(
            df_local=df,
            target=target,
            estimator=estimator,
            features_used=features,
            allowed_regions=None
        )

        dt = time.time() - t0

        preproc_path, model_path = fit_full_and_save(
            df_local=df,
            target=target,
            estimator=estimator,
            features_used=features,
            run_name=run_name
        )

        mlflow.log_param('stage', 'full')
        mlflow.log_param('target', target)
        mlflow.log_param('model_name', model_name)
        mlflow.log_param('feature_set_name', feature_set_name)
        mlflow.log_param('split_strategy', SPLIT_STRATEGY)
        mlflow.log_param('group_definition_version', GROUP_DEFINITION_VERSION)
        mlflow.log_param('group_values_hash', compute_group_values_hash(df['Region'].astype(str)))
        mlflow.log_param('feature_set_hash', compute_feature_set_hash(features))
        mlflow.log_param('pipeline_version', PIPELINE_VERSION)
        mlflow.log_param('preprocess_version', PREPROCESS_VERSION)
        mlflow.log_param('n_features_used', len(features))

        mlflow.log_metric('r2', out['r2'])
        mlflow.log_metric('mean_fold_r2', out['mean_fold_r2'])
        mlflow.log_metric('min_fold_r2', out['min_fold_r2'])
        mlflow.log_metric('rmse', out['rmse'])
        mlflow.log_metric('mae', out['mae'])
        mlflow.log_metric('cv_time_sec', dt)

        mlflow.log_artifact(preproc_path, artifact_path='submission_assets')
        mlflow.log_artifact(model_path, artifact_path='submission_assets')

        OOF_PREDS[run_name] = out['pred']

        selection_score = compute_selection_score(
            overall_r2=out['r2'],
            eastern_cape_r2=out['eastern_cape_r2'],
            min_fold_r2=out['min_fold_r2']
        )

        rows_full.append({
            'stage': 'full',
            'run_name': run_name,
            'target': target,
            'model_name': model_name,
            'feature_set': feature_set_name,
            'features_used_json': json.dumps(features),
            'r2': out['r2'],
            'mean_fold_r2': out['mean_fold_r2'],
            'min_fold_r2': out['min_fold_r2'],
            'eastern_cape_r2': out['eastern_cape_r2'],
            'selection_score': selection_score,
            'rmse': out['rmse'],
            'mae': out['mae'],
            'cv_time_sec': dt,
            'preproc_path': preproc_path,
            'model_path': model_path,
        })

    print(out['fold_df'])
    print(
        f"FULL OOF R2={out['r2']:.4f} | "
        f"Eastern_Cape R2={out['eastern_cape_r2']:.4f} | "
        f"selection_score={selection_score:.4f} | "
        f"min_fold_r2={out['min_fold_r2']:.4f}"
    )

full_df = pd.DataFrame(rows_full).sort_values(
    ['target', 'selection_score', 'r2', 'min_fold_r2'],
    ascending=[True, False, False, False]
).reset_index(drop=True)

print('\nFull results:')
display(full_df)


Finalists (union shortlist):


,target,feature_set,model_name,selection_score,eastern_cape_r2,r2,min_fold_r2,run_name
0,Dissolved Reactive Phosphorus,B,XGB_d3_lr003_Log,0.020289,0.032403,0.019286,-0.017751,SCOUT__XGB_d3_lr003_Log__B__DissolvedReactiveP...
1,Dissolved Reactive Phosphorus,C,RF_n600_Log,-0.031233,-0.011988,0.004188,-0.178030,SCOUT__RF_n600_Log__C__DissolvedReactivePhosph...
2,Electrical Conductance,B,XGB_d4_lr003_Log,-0.218445,-0.287829,-0.089588,-0.287829,SCOUT__XGB_d4_lr003_Log__B__ElectricalConductance
3,Electrical Conductance,C,RF_n600_Log,-0.693175,-0.883441,-0.339822,-0.883441,SCOUT__RF_n600_Log__C__ElectricalConductance
4,Total Alkalinity,C,RF_n600_raw,-0.362615,-0.187456,-0.337051,-1.006131,SCOUT__RF_n600_raw__C__TotalAlkalinity
5,Total Alkalinity,B,XGB_d4_lr003_Log,-0.370384,-0.282634,-0.172050,-1.125661,SCOUT__XGB_d4_lr003_Log__B__TotalAlkalinity



=== FULL__XGB_d3_lr003_Log__B__DissolvedReactivePhosphorus ===
           group     n        r2       rmse        mae
0  Northern_Bulk  5835 -0.381237  67.453993  40.805303
1   Western_Cape  1230  0.008446  28.411662  15.629235
2   Eastern_Cape  1854 -0.022765  32.090753  18.040575
3   Karoo_Buffer   400 -0.205956  18.120903  11.309598
FULL OOF R2=-0.2216 | Eastern_Cape R2=-0.0228 | selection_score=-0.1461 | min_fold_r2=-0.3812

=== FULL__RF_n600_Log__C__DissolvedReactivePhosphorus ===
           group     n        r2       rmse        mae
0  Northern_Bulk  5835 -0.375844  67.322165  40.731786
1   Western_Cape  1230 -0.175819  30.939179  20.965754
2   Eastern_Cape  1854 -0.077765  32.942313  18.908710
3   Karoo_Buffer   400 -0.368017  19.300115  11.393154
FULL OOF R2=-0.2299 | Eastern_Cape R2=-0.0778 | selection_score=-0.1757 | min_fold_r2=-0.3758

=== FULL__XGB_d4_lr003_Log__B__ElectricalConductance ===
           group     n        r2        rmse         mae
0  Northern_Bulk  5835 -

,stage,run_name,target,model_name,feature_set,features_used_json,r2,mean_fold_r2,min_fold_r2,eastern_cape_r2,selection_score,rmse,mae,cv_time_sec,preproc_path,model_path
0,full,FULL__XGB_d3_lr003_Log__B__DissolvedReactivePh...,Dissolved Reactive Phosphorus,XGB_d3_lr003_Log,B,"[""nir"", ""green"", ""swir22"", ""NDMI"", ""MNDWI"", ""p...",-0.221565,-0.150378,-0.381237,-0.022765,-0.146116,56.342544,31.687302,0.512095,../models/final_deadline\FULL__XGB_d3_lr003_Lo...,../models/final_deadline\FULL__XGB_d3_lr003_Lo...
1,full,FULL__RF_n600_Log__C__DissolvedReactivePhosphorus,Dissolved Reactive Phosphorus,RF_n600_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.229872,-0.249361,-0.375844,-0.077765,-0.175714,56.533785,32.521929,5.831413,../models/final_deadline\FULL__RF_n600_Log__C_...,../models/final_deadline\FULL__RF_n600_Log__C_...
2,full,FULL__XGB_d4_lr003_Log__B__ElectricalConductance,Electrical Conductance,XGB_d4_lr003_Log,B,"[""nir"", ""green"", ""swir22"", ""NDMI"", ""MNDWI"", ""p...",0.045485,-0.059346,-0.177283,0.024671,0.001663,334.052821,249.187833,0.829205,../models/final_deadline\FULL__XGB_d4_lr003_Lo...,../models/final_deadline\FULL__XGB_d4_lr003_Lo...
3,full,FULL__RF_n600_Log__C__ElectricalConductance,Electrical Conductance,RF_n600_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.269343,-0.239125,-0.521246,-0.018743,-0.181828,385.223893,294.497989,8.097169,../models/final_deadline\FULL__RF_n600_Log__C_...,../models/final_deadline\FULL__RF_n600_Log__C_...
4,full,FULL__XGB_d4_lr003_Log__B__TotalAlkalinity,Total Alkalinity,XGB_d4_lr003_Log,B,"[""nir"", ""green"", ""swir22"", ""NDMI"", ""MNDWI"", ""p...",0.190314,0.118340,-0.039977,0.220753,0.170990,67.206714,51.950133,0.762189,../models/final_deadline\FULL__XGB_d4_lr003_Lo...,../models/final_deadline\FULL__XGB_d4_lr003_Lo...
5,full,FULL__RF_n600_raw__C__TotalAlkalinity,Total Alkalinity,RF_n600_raw,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.345060,-0.558142,-1.395541,0.162205,-0.249000,86.621372,67.938392,7.456832,../models/final_deadline\FULL__RF_n600_raw__C_...,../models/final_deadline\FULL__RF_n600_raw__C_...


## Freeze Manifest A

This chooses one safe anchor model per target from the full finalists.
For DRP, we prefer the safer linear models.

In [74]:
TARGET_GLOBAL_R2_FLOOR = {
    'Total Alkalinity': 0.00,
    'Electrical Conductance': 0.00,
    'Dissolved Reactive Phosphorus': -0.20,
}

DRP_SAFE_MODELS = [
    'XGB_d3_lr003_Log',
    'Lasso_a0.0010_Log',
    'Ridge_a30_Log',
    'RF_n600_raw',
    'RF_n600_Log',
]

In [75]:
manifest_A = {}
manifest_B = {}

for target in TARGET_COLS:
    tdf = full_df[full_df['target'] == target].copy()

    # sort by the geography-aware score first
    tdf = tdf.sort_values(
        ['selection_score', 'r2', 'min_fold_r2'],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    floor = TARGET_GLOBAL_R2_FLOOR[target]
    passed_floor = tdf[tdf['r2'] >= floor].copy()

    if target == 'Total Alkalinity':
        # TA is stable enough to trust weighted ranking directly
        chosen_A = passed_floor.iloc[0] if not passed_floor.empty else tdf.iloc[0]
        chosen_B = chosen_A

    elif target == 'Electrical Conductance':
        # EC: do not allow a heavily negative global-R2 model to become the safe anchor
        safe_pool = passed_floor.copy()

        if safe_pool.empty:
            # fallback to least-bad global result
            safe_pool = tdf.sort_values(['r2', 'selection_score'], ascending=[False, False])

        chosen_A = safe_pool.iloc[0]

        # challenger can be East-Cape-weighted, but only if it differs
        challenger_pool = tdf.copy()
        chosen_B = challenger_pool.iloc[0]

    elif target == 'Dissolved Reactive Phosphorus':
        # DRP safe anchor = safer model with best global / worst-fold behavior
        safe_pool = tdf[tdf['model_name'].isin(DRP_SAFE_MODELS)].copy()

        if safe_pool.empty:
            safe_pool = tdf.copy()

        # A = safer global-ish pick
        safe_pool_A = safe_pool.sort_values(
            ['r2', 'min_fold_r2', 'selection_score'],
            ascending=[False, False, False]
        )
        chosen_A = safe_pool_A.iloc[0]

        # B = East-Cape-biased challenger inside safe model family
        safe_pool_B = safe_pool.sort_values(
            ['selection_score', 'eastern_cape_r2', 'r2'],
            ascending=[False, False, False]
        )
        chosen_B = safe_pool_B.iloc[0]

    manifest_A[target] = chosen_A.to_dict()
    manifest_B[target] = chosen_B.to_dict()

print('=== MANIFEST A (SAFE) ===')
print(json.dumps({
    k: {
        'run_name': v['run_name'],
        'model_name': v['model_name'],
        'feature_set': v['feature_set'],
        'selection_score': float(v['selection_score']),
        'eastern_cape_r2': float(v['eastern_cape_r2']) if pd.notna(v['eastern_cape_r2']) else None,
        'r2': float(v['r2']),
        'min_fold_r2': float(v['min_fold_r2']),
    }
    for k, v in manifest_A.items()
}, indent=2))

print('\n=== MANIFEST B (CHALLENGER) ===')
print(json.dumps({
    k: {
        'run_name': v['run_name'],
        'model_name': v['model_name'],
        'feature_set': v['feature_set'],
        'selection_score': float(v['selection_score']),
        'eastern_cape_r2': float(v['eastern_cape_r2']) if pd.notna(v['eastern_cape_r2']) else None,
        'r2': float(v['r2']),
        'min_fold_r2': float(v['min_fold_r2']),
    }
    for k, v in manifest_B.items()
}, indent=2))

=== MANIFEST A (SAFE) ===
{
  "Total Alkalinity": {
    "run_name": "FULL__XGB_d4_lr003_Log__B__TotalAlkalinity",
    "model_name": "XGB_d4_lr003_Log",
    "feature_set": "B",
    "selection_score": 0.17098965182686235,
    "eastern_cape_r2": 0.2207528345038916,
    "r2": 0.1903135528564912,
    "min_fold_r2": -0.03997672616570247
  },
  "Electrical Conductance": {
    "run_name": "FULL__XGB_d4_lr003_Log__B__ElectricalConductance",
    "model_name": "XGB_d4_lr003_Log",
    "feature_set": "B",
    "selection_score": 0.001662774585233149,
    "eastern_cape_r2": 0.02467097388779249,
    "r2": 0.04548484151371346,
    "min_fold_r2": -0.17728271258975203
  },
  "Dissolved Reactive Phosphorus": {
    "run_name": "FULL__XGB_d3_lr003_Log__B__DissolvedReactivePhosphorus",
    "model_name": "XGB_d3_lr003_Log",
    "feature_set": "B",
    "selection_score": -0.1461157561138173,
    "eastern_cape_r2": -0.022764899132008853,
    "r2": -0.22156488633829308,
    "min_fold_r2": -0.38123730886273544
  

## Diagnostic For Retraining Before Making Submission

In [76]:
cols = [
    'basin_upstream_area_km2',
    'worldpop_mean_1km',
    'river_avg_discharge_cms',
    'pop_density_upstream',
    'specific_discharge'
]
display(df[cols].describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]).T)

,count,mean,std,min,50%,90%,95%,99%,99.9%,max
basin_upstream_area_km2,9319.0,24924.741120,87167.528604,1.335000e+02,4688.700000,47423.000000,80358.300000,764462.400000,786079.800000,786079.800000
worldpop_mean_1km,9319.0,2.664821,3.879033,0.000000e+00,1.199995,8.020488,10.694789,18.795643,26.793154,26.793154
river_avg_discharge_cms,9319.0,26.927536,55.863831,1.100000e-02,6.446000,75.058998,118.915001,363.572998,436.578003,436.578003
pop_density_upstream,9319.0,0.001827,0.005071,0.000000e+00,0.000256,0.003671,0.010484,0.026349,0.058574,0.058574
specific_discharge,9319.0,0.002568,0.002892,7.648720e-07,0.001271,0.006732,0.009292,0.010869,0.013457,0.013457


## Submission helper

This loads the saved artifacts for a chosen manifest entry and generates clipped predictions.

In [77]:
def predict_from_manifest_entry(entry, df_val_local):
    '''
    Predict from a frozen manifest entry using the saved
    preprocessor and model artifacts.
    '''
    feats = json.loads(entry['features_used_json'])
    pre = joblib.load(entry['preproc_path'])
    mdl = joblib.load(entry['model_path'])

    X = df_val_local[feats]
    pred = mdl.predict(pre.transform(X))
    pred = np.asarray(pred, dtype=float)

    return np.clip(pred, 0, None)

## Build Shot A / B / C

- **Shot A** = safe anchor from Manifest A
- **Shot B** = EC aggressive + DRP safe shrink
- **Shot C** = hedge blend between A and B

In [78]:
df_val = pd.read_parquet('../data/interim/master_test.parquet')
df_val = engineer_features(df_val)

tpl = pd.read_csv('../data/raw/submission_template.csv')
tpl = make_row_id_template(tpl)

# Validate that all needed features exist in validation
needed_feats = set()
for t in TARGET_COLS:
    needed_feats.update(json.loads(manifest_A[t]['features_used_json']))
    needed_feats.update(json.loads(manifest_B[t]['features_used_json']))

missing_feats = sorted([f for f in needed_feats if f not in df_val.columns])
if missing_feats:
    raise RuntimeError(f'Missing validation features: {missing_feats}')

def clip_by_train_quantile(pred, target, q_hi=0.995):
    hi = float(df[target].quantile(q_hi))
    return np.clip(np.asarray(pred, dtype=float), 0, hi)

# -------------------------
# Shot A: safe anchor
# -------------------------
shotA = tpl.copy()
for target in TARGET_COLS:
    pred_a = predict_from_manifest_entry(manifest_A[target], df_val)
    shotA[target] = clip_by_train_quantile(pred_a, target)

assert_submission_integrity(shotA, tpl, TARGET_COLS)

# -------------------------
# Shot B: challenger blend
# -------------------------
shotB = tpl.copy()

# TA stays mostly safe
shotB['Total Alkalinity'] = shotA['Total Alkalinity']

# EC: blend safe + challenger to reduce catastrophic swaps
if manifest_B['Electrical Conductance']['run_name'] != manifest_A['Electrical Conductance']['run_name']:
    ec_safe = shotA['Electrical Conductance'].values
    ec_chal = predict_from_manifest_entry(manifest_B['Electrical Conductance'], df_val)
    ec_blend = 0.35 * ec_safe + 0.65 * ec_chal
    shotB['Electrical Conductance'] = clip_by_train_quantile(ec_blend, 'Electrical Conductance')
    print('Shot B EC challenger blend:', manifest_B['Electrical Conductance']['run_name'])
else:
    shotB['Electrical Conductance'] = shotA['Electrical Conductance']
    print('Shot B EC kept from Manifest A')

# DRP: challenger + median shrink
drp_chal = predict_from_manifest_entry(manifest_B['Dissolved Reactive Phosphorus'], df_val)
drp_train_median = float(df['Dissolved Reactive Phosphorus'].median())
drp_blend = 0.55 * drp_chal + 0.45 * drp_train_median
shotB['Dissolved Reactive Phosphorus'] = clip_by_train_quantile(
    drp_blend, 'Dissolved Reactive Phosphorus'
)
print('Shot B DRP challenger:', manifest_B['Dissolved Reactive Phosphorus']['run_name'])

assert_submission_integrity(shotB, tpl, TARGET_COLS)

# -------------------------
# Shot C: hedge (middle risk)
# -------------------------
shotC = tpl.copy()
shotC['Total Alkalinity'] = clip_by_train_quantile(
    0.85 * shotA['Total Alkalinity'] + 0.15 * shotB['Total Alkalinity'],
    'Total Alkalinity'
)
shotC['Electrical Conductance'] = clip_by_train_quantile(
    0.50 * shotA['Electrical Conductance'] + 0.50 * shotB['Electrical Conductance'],
    'Electrical Conductance'
)
shotC['Dissolved Reactive Phosphorus'] = clip_by_train_quantile(
    0.45 * shotA['Dissolved Reactive Phosphorus'] + 0.55 * shotB['Dissolved Reactive Phosphorus'],
    'Dissolved Reactive Phosphorus'
)

assert_submission_integrity(shotC, tpl, TARGET_COLS)

# -------------------------
# Save
# -------------------------
stamp = datetime.now().strftime('%Y%m%d_%H%M')
pathA = f'../data/submission/submission_{stamp}_A_safe.csv'
pathB = f'../data/submission/submission_{stamp}_B_challenger_blend.csv'
pathC = f'../data/submission/submission_{stamp}_C_hedge.csv'

os.makedirs('../data/submission', exist_ok=True)

shotA.drop(columns=['row_id']).to_csv(pathA, index=False)
shotB.drop(columns=['row_id']).to_csv(pathB, index=False)
shotC.drop(columns=['row_id']).to_csv(pathC, index=False)

print('Saved files:')
print('A:', pathA)
print('B:', pathB)
print('C:', pathC)

Shot B EC kept from Manifest A
Shot B DRP challenger: FULL__XGB_d3_lr003_Log__B__DissolvedReactivePhosphorus
Saved files:
A: ../data/submission/submission_20260312_1832_A_safe.csv
B: ../data/submission/submission_20260312_1832_B_challenger_blend.csv
C: ../data/submission/submission_20260312_1832_C_hedge.csv


## Submission diagnostics

This final cell prints simple distribution summaries for the three submission variants.

In [79]:
def summarize_shot(shot_df, name):
    print(f'\n{name} stats')
    stats = shot_df[TARGET_COLS].describe(
        percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
    ).T
    display(stats[['min', '1%', '5%', '50%', 'mean', '95%', '99%', 'max']])

summarize_shot(shotA, 'Shot A')
summarize_shot(shotB, 'Shot B')
summarize_shot(shotC, 'Shot C')

print('\nMean absolute deltas vs Shot A')
delta_tbl = pd.DataFrame({
    'target': TARGET_COLS,
    'B_vs_A_mae': [float(np.mean(np.abs(shotB[t] - shotA[t]))) for t in TARGET_COLS],
    'C_vs_A_mae': [float(np.mean(np.abs(shotC[t] - shotA[t]))) for t in TARGET_COLS],
})
display(delta_tbl)

tracker = pd.DataFrame([
    {'file': pathA, 'hypothesis': 'Safety anchor (lowest variance)'},
    {'file': pathC, 'hypothesis': 'Balanced hedge between A and B'},
    {'file': pathB, 'hypothesis': 'Most aggressive on EC/DRP challenger blend'},
])

print('\nSubmission tracker:')
display(tracker)

print('\nSuggested upload order: A -> C -> B')


Shot A stats


,min,1%,5%,50%,mean,95%,99%,max
Total Alkalinity,13.198025,26.015306,29.328127,60.725183,66.072442,146.109953,169.846849,180.237457
Electrical Conductance,105.549446,112.207654,117.714803,279.766144,312.345208,595.956067,711.255640,793.762146
Dissolved Reactive Phosphorus,11.914917,12.133108,12.759013,19.470349,22.866611,34.635423,57.549519,58.136471



Shot B stats


,min,1%,5%,50%,mean,95%,99%,max
Total Alkalinity,13.198025,26.015306,29.328127,60.725183,66.072442,146.109953,169.846849,180.237457
Electrical Conductance,105.549446,112.207654,117.714803,279.766144,312.345208,595.956067,711.255640,793.762146
Dissolved Reactive Phosphorus,15.553204,15.673210,16.017457,19.708692,21.576636,28.049483,40.652235,40.975059



Shot C stats


...


Mean absolute deltas vs Shot A


,target,B_vs_A_mae,C_vs_A_mae
0,Total Alkalinity,0.000000,1.438849e-15
1,Electrical Conductance,0.000000,0.000000e+00
2,Dissolved Reactive Phosphorus,3.213897,1.767643e+00



Submission tracker:


,file,hypothesis
0,../data/submission/submission_20260312_1832_A_...,Safety anchor (lowest variance)
1,../data/submission/submission_20260312_1832_C_...,Balanced hedge between A and B
2,../data/submission/submission_20260312_1832_B_...,Most aggressive on EC/DRP challenger blend



Suggested upload order: A -> C -> B
